# EpiSelect 124M on a Kaggle TPU (v5e-8 or v6e-8)

Trains the paper's 124M model (60k steps x 256 x 1024 tokens) with the own-weight-1 scaling-law fit, or the paper's
fit, on the compact Common Pile corpus. Accelerator: **TPU v5e-8 or v6e-8**, whichever Kaggle offers. Run with **Save Version -> Save & Run All**
(background, up to 9 h). See `kaggle/README.md` for the whole procedure.

Set `MODE` in the next cell:
- `bench`: 300 steps; prints s/step, the projected hours for 60k steps and peak HBM. Uses synthetic data if no corpus is attached.
- `train`: trains or resumes. Stops cleanly after `time_budget_h` (8.4 h). Rerun as a new version with this notebook's previous output attached until `run_status.json` says finished.
- `eval`: zero-shot LM-Eval (Table 1 tasks) and Common Pile val on the finished run.
- `report`: the gamma comparison (`scripts/gamma_report.py`), plus the Table 1 comparison if eval has run.

In [ ]:
MODE = "bench"                          # bench | train | eval | report
EXPERIMENT = "124M/EpiSelect_kaggle"    # 124M/EpiSelect_paper_kaggle for the paper-fit arm
BRANCH = "124M-run"
REPO = "https://github.com/shareesiv/episelect.git"
DATA = "/kaggle/input"                  # compact corpus: domains are found anywhere below this
RUNDIR = "/kaggle/working/run"
EXTRA = ""                              # extra hydra overrides, e.g. "g_accum_iters=4 batch_size=64"

In [ ]:
import os
import subprocess

if not os.path.isdir("/kaggle/working/EpiSelect"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/kaggle/working/EpiSelect"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/EpiSelect", "pull", "-q", "--ff-only"], check=True)
os.chdir("/kaggle/working/EpiSelect")
!git log --oneline -1
!pip install -q -e ".[tpu]" matplotlib 2>&1 | tail -3
!python -c "import jax; print(jax.__version__, jax.devices())"

In [ ]:
# W&B is optional: add a Kaggle secret WANDB_API_KEY (Add-ons -> Secrets) or logs stay offline in ./wandb
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ.setdefault("WANDB_PROJECT", "episelect-124m-kaggle")
except Exception as e:
    print("no WANDB_API_KEY secret, logging offline:", type(e).__name__)
    os.environ["WANDB_MODE"] = "offline"

In [ ]:
# resume: copy the newest run dir found in an attached previous output of this notebook
import glob
import json
import shutil


def find_previous_run():
    cands = []
    for status in glob.glob("/kaggle/input/**/run_status.json", recursive=True):
        s = json.load(open(status))
        if os.path.exists(os.path.join(os.path.dirname(status), "cfg.json")):
            cands.append((s["last_step"], os.path.dirname(status)))
    return max(cands)[1] if cands else None

if MODE == "train" and not os.path.exists(os.path.join(RUNDIR, "cfg.json")):
    prev = find_previous_run()
    if prev:
        print("resuming from", prev, json.load(open(os.path.join(prev, "run_status.json"))))
        shutil.copytree(prev, RUNDIR)
    else:
        print("no previous run attached: starting from step 0")
if MODE in ("eval", "report") and not os.path.exists(os.path.join(RUNDIR, "cfg.json")):
    prev = find_previous_run()
    assert prev, "attach the finished run (a previous version's output) as input"
    shutil.copytree(prev, RUNDIR)  # /kaggle/input is read-only; orbax wants a writable checkpoint dir
print("RUNDIR =", RUNDIR)

In [ ]:
def has_corpus(root):
    return any("train.bin" in f for _, _, fs in os.walk(root) for f in fs)

if MODE == "bench":
    data = DATA
    if not has_corpus(DATA):
        data = "/kaggle/tmp/synthetic_corpus"
        !python scripts/build_compact_commonpile.py --out {data} --synthetic_tokens 2e6
    !python -u launch.py +experiment={EXPERIMENT} loader.data_dir={data} rundir=/kaggle/tmp/bench max_steps=300 eval_interval=100 time_budget_h=null {EXTRA} 2>&1 | grep -v "^  " | tail -40

In [ ]:
if MODE == "train":
    !python -u launch.py +experiment={EXPERIMENT} loader.data_dir={DATA} rundir={RUNDIR} {EXTRA} 2>&1 | tee -a /kaggle/working/train.log | grep -v "^  " | tail -60
    print(open(os.path.join(RUNDIR, "run_status.json")).read())

In [ ]:
if MODE == "eval":
    label = EXPERIMENT.split("/")[-1]
    !python -m src.eval --config-name eval_kaggle "runs=[{{label: {label}, path: {RUNDIR}}}]" data_dir={DATA} output=/kaggle/working/eval_results.json

In [ ]:
if MODE == "report":
    ev = next(iter(glob.glob("/kaggle/input/**/eval_results.json", recursive=True)), None)
    ev = "/kaggle/working/eval_results.json" if os.path.exists("/kaggle/working/eval_results.json") else ev
    paper_run = None  # set to the paper-fit run dir if that arm was trained (attach its output)
    args = f"{RUNDIR} --replay_paper --heldout --out /kaggle/working/report"
    args += f" --paper {paper_run}" if paper_run else ""
    args += f" --eval {ev}" if ev else ""
    !python scripts/gamma_report.py {args}